# 03 Modifying the machine: element settings, groups, RF, errors

Each code cell below is one of the five scripts of `03_modifying/`, in order: the same code as the command line (`python 03_modifying/NN_name.py`), plus a first line setting `__file__`, which the scripts use to find `output/`. Change a cell to experiment; to change an example for good, edit its script and run `python sync_notebooks.py`, which copies the scripts into the notebooks. Each script's docstring says what it shows, what it needs and what it checks.

15, 17, 18 and the solenoid in 19 use small lattices built in Python (with `02_tracking/simple_lattice.py`). 16 and the dogleg in 19 run JFEL's bunch compressor as a line of its own, from a `.def` written into `output/` by `chicane.py`. Every check compares a tracked beam with a formula worked out from first principles.

Everything here is tier A (pip only); 18 adds an elegant check when the simcodes image is there. The whole notebook takes under a minute.

## 15 Read and change element settings

`framework["QD0"]` is the LAURA element. `getElement`/`modifyElement` read and set one parameter, by short name (`"k1l"`) or full path (`"magnetic.k1l"`); `getElementType`/`setElementType` do every element of a type; `apply_changes` takes a dictionary. `save_changes_file` writes what changed (old and new values) and `load_changes_file` applies it to a fresh lattice.

`getElementType` lists elements in load order, not lattice order, and `setElementType` pairs its values with that order, so build the values from `getElementType`.

The check flips every quadrupole: the new strengths must appear in the Ocelot deck, and the horizontal beam must now follow the original lattice's vertical matrix.

In [ ]:
__file__ = "15_modify_elements.py"  # python and %run set this; a notebook cell does not
"""
15 - Read and change element settings
=====================================

Shows: the ways to read and change an element's settings once a lattice is
loaded, and how to save the changes to a file and apply them again later:

- `framework["QD0"]` is the LAURA element; `.k1l` reads its strength.
- `getElement(name, param)` and `modifyElement(name, param, value)` read and
  set one parameter. `param` can be a dotted path into the element
  ("magnetic.k1l", "physical.error.position.x").
- `getElementType(type, param)` and `setElementType(type, param, values)` do
  the same for every element of a type.
- `apply_changes({name: {param: value}})` sets several at once.
- `save_changes_file(filename=...)` writes every change made since loading,
  old and new values. `load_changes_file(filename)` applies them to a fresh
  lattice.

Run:    python 03_modifying/15_modify_elements.py
Tier:   A
Needs:  SIMBA, LAURA and Ocelot.
Output: output/15_modify_elements/: changes.yaml, the lattice, and the run
        (LINE.py is the Ocelot deck SIMBA wrote).

Units: k1l in 1/m (k1 = k1l / length in 1/m^2); x in m; x' = px/pz (rad).

getElementType returns the elements in the order they were loaded, which is
not lattice order (here QD0 before QF0A). setElementType pairs its values
with that same order, so build the values from getElementType, as here,
rather than from a list in lattice order.

The script flips the sign of every quadrupole in 02_tracking's three FODO
cells, saves that as a changes file, loads it into a new Framework and
tracks.

Check: the strength SIMBA wrote into the Ocelot deck must be the new k1l
divided by the length, to 1e-12. Flipping every quadrupole swaps the
focusing planes, so the tracked horizontal sigma matrix at END must be
M_y Sigma0 M_y^T, with M_y the original lattice's vertical matrix, to 1e-4
relative (the same limit and reason as 09). The reloaded lattice must hold
exactly the values saved, and the changes file must list only the changed
quadrupoles.
"""

import re
import sys
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import fodo, make_beam, sigma_matrix, transfer_matrix, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"

elements = fodo(cells=3)
files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
settings = write_master_lattice(LATTICE, elements, files, "fodo.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", beta=(5.0, 5.0), alpha=(0.0, 0.0))

framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / "edit"), clean=True, verbose=False)
framework.loadSettings(settings)

# --- Reading -----------------------------------------------------------------
print(f'framework["QD0"]: a LAURA {type(framework["QD0"]).__name__}, k1l = {framework["QD0"].k1l}')
print(f'getElement("QD0", "k1l") = {framework.getElement("QD0", "k1l")}')
names = [e["name"] for e in framework.getElementType("quadrupole")]
print(f'getElementType("quadrupole") order: {names}  (load order, not lattice order)')

# --- Changing one element ----------------------------------------------------
framework.modifyElement("QD0", "k1l", -0.5)
print(f'modifyElement("QD0", "k1l", -0.5): k1l = {framework["QD0"].k1l}')
framework.modifyElement("QD0", "magnetic.k1l", -0.6)  # the same parameter, by its full path
print(f'modifyElement("QD0", "magnetic.k1l", -0.6): k1l = {framework["QD0"].k1l}')
framework.apply_changes({"QD1": {"k1l": -0.5}, "QD2": {"k1l": -0.5}})
print(f'apply_changes: QD1, QD2 k1l = {framework["QD1"].k1l}, {framework["QD2"].k1l}')
framework.apply_changes({"QD1": {"k1l": -0.6}, "QD2": {"k1l": -0.6}})  # back, before the flip

# --- Changing every quadrupole: flip the sign --------------------------------
k1l = framework.getElementType("quadrupole", "k1l")
framework.setElementType("quadrupole", "k1l", [-v for v in k1l])  # paired with getElementType's order
flipped = dict(zip(names, framework.getElementType("quadrupole", "k1l")))
print(f"flipped: {flipped}")

changes_file = OUT / "changes.yaml"
framework.save_changes_file(filename=str(changes_file))
changes = yaml.safe_load(changes_file.read_text())
print(f"\n{changes_file.name}, first entry:\n" + yaml.safe_dump({"QD0": changes["QD0"]}))

# --- A fresh lattice with the changes applied, tracked -----------------------
run = OUT / "run"
reloaded = fw.Framework(master_lattice=str(LATTICE), directory=str(run), clean=True, verbose=False)
reloaded.loadSettings(settings)
reloaded.load_changes_file(str(changes_file))
got_k1l = dict(zip(names, reloaded.getElementType("quadrupole", "k1l")))
reloaded.track()

deck = (run / "LINE.py").read_text()
deck_k1 = {m[1].upper(): float(m[0]) for m in re.findall(r"k1=([-\d.e]+), eid='(\w+)'", deck)}
lengths = {e.name: e.physical.length for e in elements if e.name in names}
end = rbf.load_file(str(run / "END.openpmd.hdf5"))
m_y = transfer_matrix(elements, "y", 0.0, elements[-1].physical.middle.z)  # the original lattice, y plane
expected = m_y @ sigma_matrix(start, "x") @ m_y.T
got = sigma_matrix(end, "x")
scale = np.sqrt(np.outer(np.diag(expected), np.diag(expected)))
worst = np.max(np.abs(got - expected) / scale)
print(f"Deck strengths: {deck_k1}")
print(f"sigma_x at END: {np.sqrt(got[0, 0]) * 1e6:.2f} um; M_y Sigma0 M_y^T gives "
      f"{np.sqrt(expected[0, 0]) * 1e6:.2f} um; worst difference {worst:.1e} (relative)")

# --- Check -----------------------------------------------------------------
quads = {e.name: e.magnetic.k1l for e in elements if e.name in names}
assert all(flipped[n] == -quads[n] for n in names), f"setElementType didn't flip every quad: {flipped}"
assert got_k1l == flipped, f"load_changes_file gave {got_k1l}, saved {flipped}"
assert set(changes) == set(names), f"changes file lists {sorted(changes)}, changed {sorted(names)}"
for name in names:
    assert abs(deck_k1[name] - flipped[name] / lengths[name]) < 1e-12, \
        f"{name}: deck k1 {deck_k1[name]}, expected {flipped[name] / lengths[name]}"
assert worst < 1e-4, f"flipped lattice's sigma_x differs from M_y Sigma0 M_y^T by {worst:.1e}"
print("\nOK: changes reach the deck, survive a save and reload, and flipping the quads swaps the planes")

## 16 A chicane as a group

A `.def`'s `groups:` block ties elements together. A `chicane` group has `.angle`, an `.r56` estimate and `set_angle(a)`, which re-lays out the dipoles and everything between them. The script tracks a chirp-free beam through JFEL's chicane at 0.13 and 0.10 rad, measures $R_{56}$ by fitting each particle's change in $z$ against its starting $x$, $x'$ and $\delta$, and compares with the exact formula

$$R_{56} = 2\left[2\rho(\tan\theta - \theta) + D\,\frac{\tan^2\theta}{\cos\theta}\right] + \frac{S}{\gamma^2}$$

where the last term is the velocity spread over the line's length $S$. The group's `.r56` is the small-angle formula and reads 1.1% low. `set_angle` keeps each dipole's entrance and exit along the axis and lets the arc follow, so at the angle the chicane already has it changes nothing (it once lengthened the dipoles by 0.28% there: ISSUES.md #27).

In [ ]:
__file__ = "16_groups_chicane.py"  # python and %run set this; a notebook cell does not
"""
16 - A chicane as a group: change its angle, measure R56
========================================================

Shows: a `.def`'s `groups:` block, which ties elements together so they can
be changed as one. A `chicane` group of four dipoles has `.angle`, an
`.r56` estimate, and `set_angle(a)`, which sets all four bends and moves
every element between the dipoles onto the new path. The script runs
JFEL's bunch compressor on its own (chicane.py writes the .def) at its
design angle, 0.13 rad, and at 0.10 rad, in Ocelot, Cheetah and Xsuite.

Run:    python 03_modifying/16_groups_chicane.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices, Ocelot, Cheetah and Xsuite.
Output: output/16_groups_chicane/: bc.def, start.openpmd.hdf5, and one run
        per code and angle (ocelot_0.13/, ...).

Units: angle in rad; R56 in m (z in m, delta = dp/p); positions in m.

R56 is measured from a chirp-free 130 MeV beam (energy spread 1e-3, no
z-energy correlation): fit each particle's change in z (positive = further
ahead) against its starting x, x' and delta (chicane.linear_coefficients).
Higher energy bends less, takes the shorter path and gets ahead, so R56 is
positive in SIMBA's convention. The exact path-length R56 of a rectangular
chicane (dipoles of arc L and radius rho = L/theta, gap D along z between
dipoles 1 and 2, and 3 and 4) is

    R56_path = 2 [2 rho (tan(theta) - theta) + D tan(theta)^2 / cos(theta)]

The codes also include the velocity term: over the line's length S a
particle with more energy is faster by delta/gamma^2, so the tracked
R56 = R56_path + S/gamma^2 (8.5e-5 m of 0.0563 m here).

The group's `.r56` property is the small-angle formula
2 theta^2 (D + 2L/3), 1.1% below the exact value at 0.13 rad.

set_angle also changes the geometry: in the middle straight the screen
moves sideways to X = 2 rho (1 - cos(theta)) + D tan(theta). It keeps each
dipole's entrance and exit along the axis where the lattice puts them, and
the arc follows, so set_angle at the angle the chicane already has changes
nothing (it once took the arc for the straight length: ISSUES.md #27).

Checks: set_angle at the current angle keeps the dipoles' arc to 1e-9.
In every code and at both angles, the tracked R56 must equal
R56_path + S/gamma^2 to 2e-4 relative. The only noise is the second-order
term T566 delta^2 (T566 ~ -1.5 R56), which the fit sees as scatter of
~3e-5 relative for 5000 particles; the measured difference is at most 7e-6. The
screen's x must be X to 1e-6 m (rounding in the layout; measured 1e-9).
`.r56` must be within 2% of R56_path (the small-angle error).
"""

import sys
from pathlib import Path

import numpy as np
import simba.Modules.Beams as rbf
from chicane import DIPOLES, END, SCREEN, bend_geometry, linear_coefficients, load, write_def

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import M_E, make_beam  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
OUT.mkdir(parents=True, exist_ok=True)
CODES = ["ocelot", "cheetah", "xsuite"]

start = make_beam(OUT / "start.openpmd.hdf5", beta=(10.0, 10.0), p0=130e6, sigma_pz=130e3, particles=5000)
gamma = np.mean(start.cp.val) / M_E
def_file = write_def(OUT / "bc.def", OUT / "start")

# --- The group -----------------------------------------------------------------
framework = load(def_file, OUT / "inspect", "ocelot")
group = framework["bunch_compressor"]
print(f"bunch_compressor: {type(group).__name__} of {list(group.elements)}")
print(f"angle {group.angle} rad, r56 property {group.r56:.6f} m")

# set_angle at the angle it already has changes nothing (ISSUES.md #27, fixed)
arc = framework[DIPOLES[0]].magnetic.length
group.set_angle(group.angle)
same_arc = abs(framework[DIPOLES[0]].magnetic.length / arc - 1) <= 1e-9
print(f"set_angle({group.angle}) keeps the dipole arc {arc:.6f} m: {same_arc}")

# --- Track at two angles ----------------------------------------------------------
print(f"\n{'code':8s} {'angle':>6s} {'R56 tracked':>12s} {'expected':>10s} {'rel diff':>9s} "
      f"{'r56 prop':>9s} {'screen x':>9s} {'expected':>9s}")
problems = [] if same_arc else ["set_angle at the current angle changed the dipole arc"]
for angle in (0.13, 0.10):
    for code in CODES:
        run = OUT / f"{code}_{angle}"
        f = load(def_file, run, code, angle=None if angle == 0.13 else angle)
        f.track()
        rho, gap = bend_geometry(f, angle)
        s = f["BC"].getSValues()
        r56_path = 2 * (2 * rho * (np.tan(angle) - angle) + gap * np.tan(angle) ** 2 / np.cos(angle))
        expected = r56_path + (s[-1] - s[0]) / gamma ** 2
        end = rbf.load_file(str(run / f"{END}.openpmd.hdf5"))
        r56 = linear_coefficients(start, end.z.val - start.z.val)[2]
        x_screen = f[SCREEN].physical.middle.x
        x_expected = 2 * rho * (1 - np.cos(angle)) + gap * np.tan(angle)
        prop = f["bunch_compressor"].r56
        rel = r56 / expected - 1
        print(f"{code:8s} {angle:6.2f} {r56:12.6f} {expected:10.6f} {rel:9.1e} {prop:9.6f} "
              f"{x_screen:9.6f} {x_expected:9.6f}")
        if abs(rel) > 2e-4:
            problems.append(f"{code} at {angle}: R56 {r56:.6f}, expected {expected:.6f}")
        if abs(x_screen - x_expected) > 1e-6:
            problems.append(f"{code} at {angle}: screen x {x_screen:.6f}, expected {x_expected:.6f}")
        if abs(prop / r56_path - 1) > 0.02:
            problems.append(f"{code} at {angle}: r56 property {prop:.6f}, R56_path {r56_path:.6f}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: R56 and the chicane's geometry follow the exact formulas at both angles, in every code")

## 17 Scan an RF cavity's phase

One S-band travelling-wave cavity, scanned in phase with `modifyElement`. The mean energy gain must follow $\Delta E = V\cos(\phi - \phi_0)$ with the crest $\phi_0 = 0$ (SIMBA's convention, whatever the code's) and $V = A\,(N + 3.8)\,\ell_\mathrm{cell}/\sqrt{2}$ for a travelling-wave cavity of field amplitude $A$ (a standing-wave cavity's `field_amplitude` is its voltage). The scan avoids $\pm 90°$, where Cheetah's (and at $+90°$ Ocelot's) cavity gives NaN.

Fit the energy, not the momentum: at 100 MeV they differ by $1.3\times10^{-5}$ in the gain.

In [ ]:
__file__ = "17_rf_phase_scan.py"  # python and %run set this; a notebook cell does not
"""
17 - Scan an RF cavity's phase and find the crest
=================================================

Shows: an RF cavity built in Python (LAURA `RFCavity`), its `cavity.phase`
and `simulation.field_amplitude`, and a phase scan with `modifyElement`:
the beam's energy gain against phase, fitted to find the crest and the
cavity's voltage. Run in Ocelot, Cheetah and Xsuite.

Run:    python 03_modifying/17_rf_phase_scan.py
Tier:   A
Needs:  SIMBA, LAURA, Ocelot, Cheetah and Xsuite.
Output: output/17_rf_phase_scan/: the lattice, one run directory per code,
        and phase_scan.png.

Units: phase in degrees; field_amplitude in V/m (travelling wave) or V
(standing wave); energy in eV; frequency in Hz.

The cavity: a 1 m S-band (2998.5 MHz) travelling-wave structure, 30 cells
in 2pi/3 mode, field_amplitude 20 MV/m, no field map. The beam: 100 MeV,
10 um long, so all of it sees the same phase.

SIMBA's conventions:
- phase 0 is on crest (largest energy gain), whatever the code's own
  convention; SIMBA converts.
- A standing-wave cavity's field_amplitude is its voltage.
- A travelling-wave cavity's field_amplitude is a gradient, and the voltage
  is field_amplitude * (N + 3.8) * cell_length / sqrt(2), with N the number
  of cells rounded down to a multiple of 3. That is LAURA's
  RFCavity.voltage_factor (laura/translator/converters/cavity.py), which
  every code's deck uses: 15.93 MV here.
- LAURA has a second estimate, tw_cavity_energy_gain, which gives
  field_amplitude * sin(pi/3) * N * cell_length = 17.32 MV, 8.7% more.
  Only GPT uses it (to scale magnets after a cavity with no field map), so
  it does not affect these codes.

Avoid +-90 deg exactly: Cheetah's cavity matrix divides by cos(phase) and
returns NaN there (both cavity types), as does Ocelot's travelling-wave
cavity at +90 deg. The scan below steps from -165 to 165 deg by 30 deg.

Fit the energy, sqrt((cp)^2 + (mc^2)^2), not the momentum: at 100 MeV the
momentum gain is (1 + (mc^2)^2/(2E^2)) times the energy gain, which puts
1.3e-5 on V and a curvature on the scan.

Check: in every code the mean energy gain must follow
dE = V cos(phase - phi0) with the fit's worst residual below 1e-4 of V.
Ocelot and Cheetah fit it exactly (residuals of 1e-8 eV, rounding). Xsuite's
residual is 5e-5 of V (0.8 keV) and odd in phase: it follows the beam's arrival time through the cavity, and at
100 MeV the beam is not quite at the speed of light, so a particle that
gains energy moves forward in phase by a little. It falls as 1/E^3
(measured: 27 times smaller at 300 MeV). The crest phi0 must be 0 to
0.01 deg, and V must equal the voltage_factor formula to 1e-4 (measured
2e-7).
"""

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura.models.element import RFCavity

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import AREA, M_E, make_beam, marker, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
LATTICE = OUT / "lattice"
CODES = ["ocelot", "cheetah", "xsuite"]
GRADIENT, N_CELLS, CELL = 20e6, 30, 1 / 30

cavity = RFCavity(
    name="CAV", machine_area=AREA, physical={"length": 1.0, "middle": {"z": 1.0}},
    cavity={"frequency": 2998.5e6, "phase": 0.0, "n_cells": N_CELLS, "cell_length": CELL,
            "structure_type": "TravellingWave", "mode_numerator": 1, "mode_denominator": 3},
    simulation={"field_amplitude": GRADIENT},
)
files = {
    "LINE": {
        "code": CODES[0],
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
settings = write_master_lattice(LATTICE, [marker("START", 0.0), cavity, marker("END", 2.0)], files, "rf.def")
start = make_beam(LATTICE / "START.openpmd.hdf5", particles=1000, sigma_z=10e-6)


def energy(beam):
    """Mean total energy (eV)."""
    return np.mean(np.hypot(beam.cp.val, M_E))


e0 = energy(start)

voltage = GRADIENT * (N_CELLS - N_CELLS % 3 + 3.8) * CELL / np.sqrt(2)
phases = np.arange(-165.0, 166.0, 30.0)
fit_matrix = np.c_[np.cos(np.radians(phases)), np.sin(np.radians(phases))]

print(f"Expected voltage (voltage_factor): {voltage / 1e6:.4f} MV\n")
print(f"{'code':8s} {'V [MV]':>9s} {'V/expected-1':>13s} {'crest [deg]':>12s} {'worst resid / V':>16s}")
scans, problems = {}, []
for code in CODES:
    framework = fw.Framework(master_lattice=str(LATTICE), directory=str(OUT / code), clean=True, verbose=False)
    framework.loadSettings(settings)
    framework.change_Lattice_Code("LINE", code)
    gain = []
    for phase in phases:
        framework.modifyElement("CAV", "phase", float(phase))
        framework.track()
        gain.append(energy(rbf.load_file(str(OUT / code / "END.openpmd.hdf5"))) - e0)
    gain = np.array(gain)
    (a, b), *_ = np.linalg.lstsq(fit_matrix, gain, rcond=None)  # V cos(p - p0) = a cos p + b sin p
    v, crest = np.hypot(a, b), np.degrees(np.arctan2(b, a))
    resid = np.max(np.abs(fit_matrix @ [a, b] - gain))
    scans[code] = gain
    print(f"{code:8s} {v / 1e6:9.4f} {v / voltage - 1:13.1e} {crest:12.4f} {resid / v:16.1e}")
    if resid > 1e-4 * voltage or abs(crest) > 0.01 or abs(v / voltage - 1) > 1e-4:
        problems.append(f"{code}: V {v:.1f} (expected {voltage:.1f}), crest {crest:.4f} deg, residual {resid:.1e} eV")

fine = np.linspace(-180, 180, 361)
for code, gain in scans.items():
    plt.plot(phases, gain / 1e6, "o", label=code, alpha=0.7)
plt.plot(fine, voltage / 1e6 * np.cos(np.radians(fine)), "k-", lw=0.8, label="V cos(phase)")
plt.xlabel("phase [deg]")
plt.ylabel("energy gain [MeV]")
plt.legend()
plt.savefig(OUT / "phase_scan.png", dpi=120, bbox_inches="tight")
print(f"\nPlot: {OUT / 'phase_scan.png'}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: every code puts the crest at 0 deg and gives the voltage_factor voltage")

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/17_rf_phase_scan/phase_scan.png"))

## 18 Misaligned quadrupoles and random errors

`physical.error.position.x` (and `.y`, `.z`, `rotation.*`) moves an element off axis, in every code. A quad displaced by $d$ acts on $x - d$, so the centroid map is $u \to M_q(u - d) + d$. The script checks that for one quad in four codes, then for seeded random offsets on every quad of three FODO cells.

SIMBA's own `loadElementErrors`/`setSeedValue` work only in elegant and only on strengths; on main they raise `KeyError` (`XFAIL #25`, printed only when the simcodes image is there).

In [ ]:
__file__ = "18_errors_and_misalignments.py"  # python and %run set this; a notebook cell does not
"""
18 - Misaligned quadrupoles and random errors
=============================================

Shows: moving an element off axis with `physical.error.position.{x,y}`
(also `rotation.{theta,phi,psi}`, and `.z`), set with modifyElement, and
what it does to the beam's centroid:

1. One quadrupole offset, in Ocelot, Cheetah, Xsuite and MAD-X.
2. Random offsets on every quadrupole of three FODO cells, drawn from a
   seeded numpy generator: the orbit they give, and that the same seed
   gives the same orbit.
3. SIMBA's own random errors, `loadElementErrors` and `setSeedValue`.
   They are written into elegant's `&error_element`, so they work only in
   elegant, and only for strengths (k1l, ...), not positions. On SIMBA
   main they fail with KeyError for every element (ISSUES.md #25); this
   part runs only when the simcodes image is there, and prints XFAIL.

Run:    python 03_modifying/18_errors_and_misalignments.py
Tier:   A (part 3 needs tier B)
Needs:  SIMBA, LAURA, Ocelot, Cheetah, Xsuite and MAD-X (cpymad); docker
        with the simcodes image for part 3.
Output: output/18_errors_and_misalignments/: one run directory per case.

Units: offsets and x in m; x' = px/pz (rad); k1l in 1/m.

A quadrupole displaced by d acts on x - d: its centroid map is
u -> Mq (u - d) + d, with u = (x, x') and Mq the quad's matrix. For a thin
quad that is a kick k1l * d. With drifts D1 before and D2 after,
u_end = D2 [Mq (D1 u0 - d) + d]. Through many quads, apply that one quad at
a time (orbit() below).

Check: in each code the beam's centroid at END must follow the map above,
in x and y, to 1e-3 of the centroid's change. The maps are first order and
use the reference momentum, so the beam's energy spread and second-order
terms make a small difference (measured 4e-6 in Ocelot, Cheetah and
Xsuite); MAD-X tracks a sliced, thin-lens copy of the quad, which gives
5e-5. For the random offsets (Ocelot, 50 um rms), the orbit at every screen
must follow the map to 1e-3 of the largest orbit offset (measured 6e-6); a
second run with the same seed must
give exactly the same orbit, and another seed an orbit that differs by
more than 10% of it.
"""

import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura.models.element import Quadrupole

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import fodo, make_beam, marker, quad, transfer_matrix, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


def orbit(elements, offsets, plane, u0, z_to):
    """Centroid (u, u') at z_to from u0 at z = 0, with quadrupole `name` displaced by offsets[name]."""
    u, z = np.asarray(u0, dtype=float), 0.0
    quads = sorted((e for e in elements if isinstance(e, Quadrupole)), key=lambda e: e.physical.middle.z)
    for q in quads:
        start = q.physical.middle.z - q.physical.length / 2
        end = start + q.physical.length
        if end > z_to + 1e-9:
            break
        d = np.array([offsets.get(q.name, 0.0), 0.0])
        u = transfer_matrix(elements, plane, z, start) @ u  # the drift up to the quad
        u = transfer_matrix(elements, plane, start, end) @ (u - d) + d
        z = end
    return transfer_matrix(elements, plane, z, z_to) @ u


def centroid(beam, plane):
    return np.array([np.mean(getattr(beam, plane).val), np.mean(getattr(beam, f"cp{plane}").val / beam.cpz.val)])


def framework_for(lattice, settings, run, code, **kwargs):
    f = fw.Framework(master_lattice=str(lattice), directory=str(run), clean=True, verbose=False, **kwargs)
    f.loadSettings(settings)
    f.change_Lattice_Code("LINE", code)
    return f


files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
problems = []

# --- 1. One quad, offset -------------------------------------------------------
lattice = OUT / "one_quad" / "lattice"
elements = [marker("START", 0.0), quad("Q1", 0.5, 0.5), marker("END", 2.5)]
settings = write_master_lattice(lattice, elements, files, "quad.def")
start = make_beam(lattice / "START.openpmd.hdf5", particles=2000)
offset = {"x": 1e-4, "y": -2e-4}
print(f"Q1 (k1l = 0.5 /m) offset by x {offset['x']:.0e} m, y {offset['y']:.0e} m; "
      f"thin-lens kick k1l*dx = {0.5 * offset['x']:.1e} rad")
print(f"{'code':8s} {'x_end [um]':>11s} {'expected':>9s} {'y_end [um]':>11s} {'expected':>9s} {'worst rel':>9s}")
for code in ["ocelot", "cheetah", "xsuite", "madx"]:
    run = OUT / "one_quad" / code
    f = framework_for(lattice, settings, run, code)
    for plane, d in offset.items():
        f.modifyElement("Q1", f"physical.error.position.{plane}", d)
    f.track()
    end = rbf.load_file(str(run / "END.openpmd.hdf5"))
    row, worst = f"{code:8s}", 0.0
    for plane, d in offset.items():
        u0 = centroid(start, plane)
        expected = orbit(elements, {"Q1": d}, plane, u0, 2.5)
        aligned = orbit(elements, {}, plane, u0, 2.5)
        got = centroid(end, plane)
        worst = max(worst, abs(got[0] - expected[0]) / abs(expected[0] - aligned[0]))
        row += f" {got[0] * 1e6:11.3f} {expected[0] * 1e6:9.3f}"
    print(row + f" {worst:9.1e}")
    if worst > 1e-3:
        problems.append(f"{code}: centroid differs from the offset-quad map by {worst:.1e} of its change")

# --- 2. Random offsets on every quad, seeded ---------------------------------------
lattice = OUT / "fodo" / "lattice"
elements = fodo(cells=3)
settings = write_master_lattice(lattice, elements, files, "fodo.def")
start = make_beam(lattice / "START.openpmd.hdf5", particles=2000)
names = [e.name for e in elements if isinstance(e, Quadrupole)]
screens = [e for e in elements if e.hardware_class == "Marker" and e.name != "START"]


def random_orbit(seed, tag):
    """Offset every quad by a seeded random amount (50 um rms), track, return the orbit at each screen."""
    rng = np.random.default_rng(seed)
    offsets = {plane: dict(zip(names, rng.normal(0.0, 50e-6, len(names)))) for plane in ("x", "y")}
    f = framework_for(lattice, settings, OUT / "fodo" / tag, "ocelot")
    for plane in ("x", "y"):
        for name, d in offsets[plane].items():
            f.modifyElement(name, f"physical.error.position.{plane}", float(d))
    f.track()
    got, expected = [], []
    for screen in screens:
        beam = rbf.load_file(str(OUT / "fodo" / tag / f"{screen.name}.openpmd.hdf5"))
        for plane in ("x", "y"):
            got.append(centroid(beam, plane)[0])
            expected.append(orbit(elements, offsets[plane], plane, centroid(start, plane), screen.physical.middle.z)[0])
    return np.array(got), np.array(expected)


orbit_1, expected_1 = random_orbit(1, "seed1")
orbit_1b, _ = random_orbit(1, "seed1_again")
orbit_2, _ = random_orbit(2, "seed2")
scale = np.max(np.abs(expected_1))
worst = np.max(np.abs(orbit_1 - expected_1)) / scale
print(f"\nRandom offsets, seed 1: largest orbit offset {scale * 1e6:.1f} um; worst difference from the map "
      f"{worst:.1e} of it")
print(f"Same seed again: largest difference {np.max(np.abs(orbit_1b - orbit_1)):.1e} m; "
      f"seed 2: {np.max(np.abs(orbit_2 - orbit_1)) * 1e6:.1f} um")
if worst > 1e-3:
    problems.append(f"random offsets: orbit differs from the map by {worst:.1e} of its size")
if not np.array_equal(orbit_1b, orbit_1):
    problems.append("the same seed gave a different orbit")
if np.max(np.abs(orbit_2 - orbit_1)) < 0.1 * scale:
    problems.append("seed 2 gave nearly the same orbit as seed 1")

# --- 3. SIMBA's random errors (elegant only) ------------------------------------------
if docker_ready():
    f = framework_for(lattice, settings, OUT / "elegant_errors", "elegant", container_runtime="docker")
    f.loadElementErrors({"elements": {"QD0": {"k1l": {"amplitude": 0.05, "fractional": 1}}}, "nruns": 2, "seed": 1})
    try:
        f.track()
    except KeyError as exc:
        if exc.args != ("Quadrupole",):
            raise
        print("\nXFAIL #25: loadElementErrors in elegant raises KeyError 'Quadrupole'")
    else:
        print("\nXPASS #25: loadElementErrors ran in elegant; check the seeds here and update ISSUES.md")
else:
    print("\nPart 3 skipped: no simcodes image (elegant is needed for loadElementErrors)")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: offset quads steer the beam as the maps say, in every code; the same seed gives the same orbit")

## 19 Dispersion from a dogleg, rotation in a solenoid

The first half of JFEL's chicane is a dogleg. Its rectangular dipoles return every particle parallel to the axis, so at the middle screen $\eta' = 0$ exactly and

$$\eta = 2\rho\left(\frac{1}{\cos\theta} - 1\right) + D\,\frac{\tan\theta}{\cos^2\theta}.$$

A solenoid rotates a beam by the Larmor angle $S_{0L}/(2B\rho)$; a flat, nearly parallel beam shows it as a change of tilt.

In [ ]:
__file__ = "19_dispersion_and_solenoid.py"  # python and %run set this; a notebook cell does not
"""
19 - Dispersion from a dogleg, rotation in a solenoid
=====================================================

Shows: two linear-optics effects, each measured on a tracked beam and
compared with its formula.

1. Dispersion. The first half of JFEL's chicane (dipoles 1 and 2) is a
   dogleg: it moves the beam sideways by X and leaves it parallel to the
   axis again. A particle with more energy bends less, so it is moved less.
   At the screen in the middle straight, x depends on the energy as
   x = eta delta, and x' doesn't (eta' = 0). Run in Ocelot, Cheetah and
   Xsuite, using 16's chicane .def (chicane.py).
2. A solenoid (LAURA `Solenoid`, built in Python) rotates the beam about
   its axis by S0L / (2 B rho), half the angle K L with K = B / (B rho):
   the Larmor angle. Run in Ocelot, Cheetah, Xsuite and MAD-X.

Run:    python 03_modifying/19_dispersion_and_solenoid.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices, Ocelot, Cheetah, Xsuite and MAD-X
        (cpymad).
Output: output/19_dispersion_and_solenoid/: the chicane .def and beam,
        dogleg_<code>/, and solenoid/ (lattice and one run per code).

Units: eta in m (x per unit delta = dp/p); angles in rad; S0L (the
solenoid's integrated field) in T m; B rho in T m.

The dogleg: JFEL's dipoles are rectangular magnets, with both faces square
to the z axis. A particle of momentum p0 (1 + delta) has radius
rho (1 + delta) and crosses the first magnet's z length rho sin(theta)
while turning by theta' with sin(theta') = sin(theta) / (1 + delta). The
second magnet, the same length in z, turns it back by exactly theta', so
every particle leaves parallel to z (eta' = 0 to all orders), displaced by

    X(delta) = 2 rho (1 + delta) (1 - cos(theta')) + D tan(theta')

where D is the z gap between the two magnets. X(0) is where the screen
sits (checked in 16). The codes' x points away from the bend's centre of
curvature, so x = X(0) - X(delta) and

    eta = -dX/ddelta = 2 rho (1/cos(theta) - 1) + D tan(theta) / cos(theta)^2

eta is measured like 16's R56: fit x (and x') at the screen against the
starting x, x' and delta.

The solenoid: 0.4 m, S0L = 0.2 T m, a 100 MeV beam (B rho = 0.3336 T m),
so the rotation is 0.2998 rad. The beam is flat (enx 10 um, eny 0.1 um)
and nearly parallel (beta = 1000 m), so its tilt,
1/2 atan2(2<xy>, <x^2> - <y^2>), is the rotation. The starting beam has a
small tilt of its own from sampling (-0.0026 rad), so measure the change.

Check: in every code, eta must match the formula to 1e-4 relative (the
fit's noise from the delta^2 term; measured 5e-6), and |eta'| must be below
1e-5 (measured 3e-9). The beam's tilt must change by S0L / (2 B rho) to
1e-3 relative, with the same sign in every code (positive S0L turns the
beam towards +y here). The solenoid also focuses, which changes the shape
of a beam that isn't perfectly parallel, and so its tilt a little:
measured 4e-5 at most (Xsuite), 3e-6 in the others.
"""

import sys
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from chicane import SCREEN, bend_geometry, linear_coefficients, load, write_def
from laura.models.element import Solenoid

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
from simple_lattice import AREA, make_beam, marker, write_master_lattice  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
OUT.mkdir(parents=True, exist_ok=True)
C = 299792458.0
problems = []

# --- 1. Dispersion at the middle of the chicane ------------------------------
start = make_beam(OUT / "start.openpmd.hdf5", beta=(10.0, 10.0), p0=130e6, sigma_pz=130e3, particles=5000)
def_file = write_def(OUT / "bc.def", OUT / "start")
theta = 0.13
print(f"{'code':8s} {'eta [m]':>9s} {'expected':>9s} {'rel diff':>9s} {'eta prime':>10s}")
for code in ["ocelot", "cheetah", "xsuite"]:
    run = OUT / f"dogleg_{code}"
    f = load(def_file, run, code)
    f.track()
    rho, gap = bend_geometry(f, theta)
    expected = 2 * rho * (1 / np.cos(theta) - 1) + gap * np.tan(theta) / np.cos(theta) ** 2
    screen = rbf.load_file(str(run / f"{SCREEN}.openpmd.hdf5"))
    eta = linear_coefficients(start, screen.x.val)[2]
    eta_prime = linear_coefficients(start, screen.cpx.val / screen.cpz.val)[2]
    print(f"{code:8s} {eta:9.6f} {expected:9.6f} {eta / expected - 1:9.1e} {eta_prime:10.1e}")
    if abs(eta / expected - 1) > 1e-4 or abs(eta_prime) > 1e-5:
        problems.append(f"{code}: eta {eta:.6f} (expected {expected:.6f}), eta' {eta_prime:.1e}")

# --- 2. Rotation in a solenoid -----------------------------------------------
S0L, P0 = 0.2, 100e6
brho = P0 / C
solenoid = Solenoid(name="SOL", machine_area=AREA, magnetic={"length": 0.4, "fields": {"S0L": S0L}},
                    physical={"length": 0.4, "middle": {"z": 0.5}})
files = {
    "LINE": {
        "code": "ocelot",
        "input": {"particle_definition": "$master_lattice$START"},
        "output": {"start_element": "START", "end_element": "END"},
    }
}
lattice = OUT / "solenoid" / "lattice"
settings = write_master_lattice(lattice, [marker("START", 0.0), solenoid, marker("END", 0.7)], files, "sol.def")
flat = make_beam(lattice / "START.openpmd.hdf5", beta=(1000.0, 1000.0), enx=10e-6, eny=0.1e-6, p0=P0,
                 particles=2000)


def tilt(beam):
    """Angle of the beam's long axis in the x-y plane (rad)."""
    x, y = beam.x.val - np.mean(beam.x.val), beam.y.val - np.mean(beam.y.val)
    return 0.5 * np.arctan2(2 * np.mean(x * y), np.mean(x * x) - np.mean(y * y))


larmor = S0L / (2 * brho)
print(f"\nSolenoid: S0L {S0L} T m, B rho {brho:.4f} T m, expected rotation {larmor:.5f} rad "
      f"(starting tilt {tilt(flat):.5f} rad)")
print(f"{'code':8s} {'rotation':>9s} {'rel diff':>9s}")
for code in ["ocelot", "cheetah", "xsuite", "madx"]:
    run = OUT / "solenoid" / code
    f = fw.Framework(master_lattice=str(lattice), directory=str(run), clean=True, verbose=False)
    f.loadSettings(settings)
    f.change_Lattice_Code("LINE", code)
    f.track()
    rotation = tilt(rbf.load_file(str(run / "END.openpmd.hdf5"))) - tilt(flat)
    print(f"{code:8s} {rotation:9.5f} {rotation / larmor - 1:9.1e}")
    if abs(rotation / larmor - 1) > 1e-3:
        problems.append(f"{code}: solenoid rotated the beam by {rotation:.5f} rad, expected {larmor:.5f}")

# --- Check -----------------------------------------------------------------
assert not problems, "\n".join(problems)
print("\nOK: the dogleg's dispersion and the solenoid's rotation match their formulas in every code")